# Session 07 — Working with TXT, CSV & JSON Files
## Python Fundamentals & Practical Applications — 48-Hour Edition

**Student edition — learn by doing**  
**Duration:** 4 classroom hours, including a 30-minute break  
**Audience:** beginners and learners with some programming experience

> **How to learn from this notebook:** Read the question, predict what will happen, run the code, explain the result in your own words, and then change something. Do not worry about making mistakes—debugging is part of programming.



In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the Python program.")

### Learning outcomes
By the end of this session, you should be able to:
1. Explain why programs save information in files.
2. Use `pathlib` to work with file paths.
3. Read, write, and append UTF-8 text.
4. Read and write CSV records using `csv.DictReader` and `csv.DictWriter`.
5. Load and save JSON data using `json.load` and `json.dump`.
6. Convert strings to numbers and diagnose common file errors.
7. Combine these skills in a small reporting project.

### Suggested 240-minute plan
| Time | Activity |
|---|---|
| 0–15 min | Welcome, goals, and prior-knowledge check |
| 15–35 min | Why files? Paths and setup |
| 35–75 min | TXT files |
| 75–125 min | CSV files |
| 125–140 min | Break |
| 140–180 min | JSON files |
| 180–205 min | Debugging and guided practice |
| 205–235 min | Mini-project |
| 235–240 min | Exit ticket |

**Important:** All examples use invented practice data. Do not put private or sensitive real-world information into shared notebooks.


## 1. Before you start: predict, run, explain

A variable stores information while a program is running. What if you want the information to still be available tomorrow, after the program has stopped?

Files let programs save information so it can be read later.

**Think first:** What information might a shop, school, hospital, or network operations team need to save? Write down one example before continuing.


In [ ]:
# Run this cell first. It creates practice folders and small synthetic datasets.
from pathlib import Path
import csv
import json

# Colab gets a predictable folder; local Jupyter uses a folder beside this notebook.
try:
    import google.colab  # type: ignore
    COURSE_ROOT = Path("/content/python_48h_course")
except ImportError:
    COURSE_ROOT = Path.cwd() / "python_48h_course"

DATA_DIR = COURSE_ROOT / "datasets"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create sample files only if they do not already exist.
def create_sample_files():
    text_file = DATA_DIR / "example_events.txt"
    if not text_file.exists():
        text_file.write_text(
            "09:00 | Router restarted\n"
            "09:12 | Connection restored\n"
            "10:05 | Customer called support\n",
            encoding="utf-8"
        )

    transactions_file = DATA_DIR / "transactions.csv"
    if not transactions_file.exists():
        rows = [
            {"transaction_id": "T001", "customer": "Ahmed", "amount": "250", "status": "Completed"},
            {"transaction_id": "T002", "customer": "Sara", "amount": "175", "status": "Pending"},
            {"transaction_id": "T003", "customer": "Omar", "amount": "400", "status": "Completed"},
            {"transaction_id": "T004", "customer": "Mona", "amount": "90", "status": "Completed"},
            {"transaction_id": "T005", "customer": "Youssef", "amount": "not available", "status": "Pending"},
        ]
        with transactions_file.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=["transaction_id", "customer", "amount", "status"])
            writer.writeheader()
            writer.writerows(rows)

    events_file = DATA_DIR / "network_events.csv"
    if not events_file.exists():
        rows = [
            {"event_id": "E001", "department": "Network", "priority": "High", "status": "Open", "hours_open": "8"},
            {"event_id": "E002", "department": "Support", "priority": "Medium", "status": "Closed", "hours_open": "2"},
            {"event_id": "E003", "department": "Network", "priority": "High", "status": "Open", "hours_open": "5"},
            {"event_id": "E004", "department": "Security", "priority": "Low", "status": "Open", "hours_open": "12"},
            {"event_id": "E005", "department": "Support", "priority": "Medium", "status": "Open", "hours_open": "3"},
            {"event_id": "E006", "department": "Network", "priority": "Low", "status": "Closed", "hours_open": "1"},
        ]
        with events_file.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=["event_id", "department", "priority", "status", "hours_open"])
            writer.writeheader()
            writer.writerows(rows)

    api_file = DATA_DIR / "api_response.json"
    if not api_file.exists():
        sample = {
            "service": "sample-status-api",
            "success": True,
            "items": [
                {"id": 101, "name": "Router", "active": True},
                {"id": 102, "name": "Switch", "active": False}
            ],
            "message": None
        }
        api_file.write_text(json.dumps(sample, indent=2), encoding="utf-8")

create_sample_files()

print("Practice data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Input files:")
for path in sorted(DATA_DIR.iterdir()):
    print(" •", path.name)


### What did the setup cell do?
- `Path` represents a filesystem path.
- `mkdir(..., exist_ok=True)` creates a folder if needed without failing when it already exists.
- `encoding="utf-8"` supports a wide range of characters.
- The files contain invented data, so you can safely experiment.

Input files are in `DATA_DIR`. Save files you create during the lesson in `OUTPUT_DIR`. This helps keep original practice data separate from your results.


## 2. Paths and TXT files

### 2.1 Inspect a path before opening a file

A **path** tells Python where a file or folder is located. A path is not the file's contents.

**Predict:** What do you think the next cell will print?


In [ ]:
text_path = DATA_DIR / "example_events.txt"

print("Path:", text_path)
print("Does the file exist?", text_path.exists())
print("Is it a file?", text_path.is_file())


### 2.2 Read a text file

`read_text()` reads the entire file and returns one string. `encoding="utf-8"` tells Python how to decode the bytes into text.

**Predict:** Will `content` be a list or a string?


In [ ]:
content = text_path.read_text(encoding="utf-8")
print(content)
print("Python type:", type(content).__name__)


**Check your understanding**
1. Why might a text file contain `\n`?
2. What does `read_text()` return?
3. Why is it useful to specify UTF-8?


### 2.3 Write a new text file

Writing is useful for saving notes, logs, or a report. `write_text()` creates a file if it does not exist. If it already exists, its old contents are replaced.

> **Caution:** Do not use write mode on a file you need to preserve unless replacing its contents is intentional.


In [ ]:
note_path = OUTPUT_DIR / "my_note.txt"

note_path.write_text(
    "My first saved note.\nPython can read it later.",
    encoding="utf-8"
)

print(note_path.read_text(encoding="utf-8"))


### 2.4 Append instead of replacing

Appending adds text to the end of an existing file. The `"a"` mode means append. The `with` statement makes sure Python closes the file when the indented block finishes, even if an error occurs.



In [ ]:
log_path = OUTPUT_DIR / "activity_log.txt"

with log_path.open("a", encoding="utf-8") as file:
    file.write("A new activity was recorded.\n")

print(log_path.read_text(encoding="utf-8"))


### 2.5 Read line by line

For large files, processing one line at a time can use less memory than reading the entire file at once.


In [ ]:
with text_path.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(line_number, line.rstrip())


### TXT quick reference

| Task | Approach | Important detail |
|---|---|---|
| Read all text | `path.read_text(encoding="utf-8")` | Returns a string |
| Replace file contents | `path.write_text(text, encoding="utf-8")` | Existing contents are overwritten |
| Append text | `path.open("a", encoding="utf-8")` | Adds content at the end |
| Read line by line | `for line in file:` | Useful for large files |
| Check path | `path.exists()` | Checks whether the path exists |

**Try it:** Create `OUTPUT_DIR / "study_notes.txt"` with two lines, then read and print it.


## 3. CSV files: tables stored as text

CSV means **comma-separated values**. It is a common format for moving table-like data between spreadsheets and programs.

Open `transactions.csv` as plain text. Notice the first line contains the column names, and each later line is one record.


In [ ]:
csv_path = DATA_DIR / "transactions.csv"
print(csv_path.read_text(encoding="utf-8"))


### 3.1 Read records using `csv.DictReader`

`DictReader` uses the header row as dictionary keys. Each row is a dictionary-like object, so you can select a value by its column name.

**Predict:** What will `row["customer"]` contain for the first record?


In [ ]:
with csv_path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    first_row = next(reader)
    print(first_row)
    print("Customer:", first_row["customer"])
    print("Amount value:", first_row["amount"])
    print("Amount type:", type(first_row["amount"]).__name__)


Notice that values read from a CSV file are strings by default—even when they look like numbers. Convert them before performing arithmetic.

### 3.2 Filter records and calculate a total

**Goal:** Add the amounts for transactions whose status is `Completed`.

Before running the code, work out which three rows should be included. One row contains a non-numeric amount, so we will also handle that safely.


In [ ]:
completed_total = 0.0
skipped_amounts = 0

with csv_path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        if row["status"] == "Completed":
            try:
                completed_total += float(row["amount"])
            except ValueError:
                skipped_amounts += 1

print("Completed transaction total:", completed_total)
print("Invalid amounts skipped:", skipped_amounts)


### 3.3 Write a CSV file

`DictWriter` writes dictionaries into a CSV file. `fieldnames` defines the column order. `writeheader()` writes the header row.

We will create a small report in `OUTPUT_DIR` so the original input file stays unchanged.


In [ ]:
report_path = OUTPUT_DIR / "completed_transactions.csv"

with csv_path.open("r", newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    completed_rows = [row for row in reader if row["status"] == "Completed"]

with report_path.open("w", newline="", encoding="utf-8") as destination:
    fieldnames = ["transaction_id", "customer", "amount", "status"]
    writer = csv.DictWriter(destination, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(completed_rows)

print(report_path.read_text(encoding="utf-8"))


### CSV quick reference

| Task | Tool |
|---|---|
| Read table records | `csv.DictReader(file)` |
| Write table records | `csv.DictWriter(file, fieldnames=...)` |
| Write column names | `writer.writeheader()` |
| Write one record | `writer.writerow(record)` |
| Write many records | `writer.writerows(records)` |
| Convert a numeric string | `float(value)` or `int(value)` |

**Common mistake:** CSV is a file format, not a Python data type. A CSV field such as `"250"` is commonly read as a string and must be converted for arithmetic.


## 4. JSON files: structured information

JSON is commonly used by web APIs and configuration files. It can represent objects, lists, strings, numbers, Boolean values, and `null`.

### 4.1 Understand a Python dictionary first


In [ ]:
student = {
    "name": "Mona",
    "age": 22,
    "skills": ["Python", "SQL"]
}

print(student["name"])
print(student["skills"])
print(student["skills"][0])


`student["skills"]` returns a list. Adding `[0]` retrieves the first item from that list. Python indexes start at zero.

### 4.2 Load JSON from a file

- `json.load(file)` reads JSON from an open file and creates a corresponding Python object.
- `json.loads(text)` reads JSON from a string.


In [ ]:
json_path = DATA_DIR / "api_response.json"

with json_path.open("r", encoding="utf-8") as file:
    response = json.load(file)

print(response)
print("Service:", response["service"])
print("First item:", response["items"][0]["name"])
print("Message:", response["message"])


### 4.3 Write JSON to a file

- `json.dump(object, file)` writes a Python object to an open file.
- `json.dumps(object)` returns a JSON-formatted string.
- `indent=2` makes the output easier for humans to read.


In [ ]:
summary = {
    "total_items": len(response["items"]),
    "active_items": sum(1 for item in response["items"] if item["active"]),
    "source": response["service"]
}

summary_path = OUTPUT_DIR / "api_summary.json"
with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(summary_path.read_text(encoding="utf-8"))


### Python values and JSON values

| Python | JSON representation |
|---|---|
| `dict` | object |
| `list` | array |
| `str` | string |
| `int` / `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

### Important distinction
`json.load()` expects an open file object. `json.loads()` expects a string. Likewise, `json.dump()` writes to a file, while `json.dumps()` returns a string.


## 5. Debugging: learn to read the error

Errors are useful clues. Read the final line of the traceback first, then look at the line of code that caused it.

### Example A — A file does not exist


In [ ]:
missing_path = DATA_DIR / "this_file_does_not_exist.txt"

if missing_path.exists():
    print(missing_path.read_text(encoding="utf-8"))
else:
    print("The file does not exist:", missing_path.name)


Checking `exists()` is useful for diagnosis, but a file can change after the check. In real applications, use appropriate exception handling when needed.


### Example B — Invalid numeric data

`float("not available")` raises `ValueError` because the text cannot be interpreted as a number.


In [ ]:
raw_amount = "not available"

try:
    amount = float(raw_amount)
    print(amount)
except ValueError:
    print("This value is not a valid number:", raw_amount)


### Example C — String concatenation versus arithmetic

Predict both outputs before running this cell.


In [ ]:
print("250" + "50")  # string concatenation
print(float("250") + 50)  # numeric addition


### Example D — Invalid JSON

Malformed JSON raises `json.JSONDecodeError`. Avoid silently ignoring malformed data; report the problem so it can be fixed.


In [ ]:
bad_json = '{"name": "Mona", "age": }'

try:
    parsed = json.loads(bad_json)
except json.JSONDecodeError as error:
    print("JSON could not be parsed.")
    print("Reason:", error.msg)


## 6. Practice exercises

Try each exercise before looking at any solution. Use hints only after making a genuine attempt.

### Exercise 1 — Foundation: save a note
Create a text file named `study_notes.txt` in `OUTPUT_DIR`. It must contain your name and one thing you learned today. Read the file and print its contents.

**Hint:** Use `Path.write_text()` and `Path.read_text()`.


In [ ]:
# Your solution:
# 1. Create the path.
# 2. Write two lines.
# 3. Read and print the saved text.



### Exercise 2 — Foundation: count text lines
Read `example_events.txt` and print how many non-empty lines it contains.

**Hint:** Loop through the file and count lines whose stripped text is not empty.


In [ ]:
# Your solution:



### Exercise 3 — Foundation: count CSV records
Read `transactions.csv` and print the number of data records (do not count the header as a record).

**Hint:** Count rows produced by `csv.DictReader`.


In [ ]:
# Your solution:



### Exercise 4 — Applied: total pending transactions
Calculate the total amount for transactions with status `Pending`. The input contains one invalid amount. Decide how your program should handle it, and print how many values were skipped.

**Hint:** Use `try` / `except ValueError`.


In [ ]:
# Your solution:



### Exercise 5 — Applied: count events by priority
Read `network_events.csv` and count how many records have each priority. Print a readable summary.

**Hint:** A dictionary can store counts. For example, `counts[key] = counts.get(key, 0) + 1`.


In [ ]:
# Your solution:



### Exercise 6 — Applied: export high-priority events
Write all high-priority records from `network_events.csv` to `high_priority_events.csv` in `OUTPUT_DIR`. Include the header row.

**Hint:** Use `DictReader` to read and `DictWriter` to write.


In [ ]:
# Your solution:



### Exercise 7 — Challenge: summarize JSON items
Load `api_response.json`. Create a new JSON file containing:
- the service name,
- total item count,
- active item count,
- inactive item count.

**Hint:** Use `json.load()`, a loop or `sum()`, and `json.dump()`.


In [ ]:
# Your solution:



### Exercise 8 — Challenge: validate transaction amounts
Read every row in `transactions.csv`. Create a report with two lists: valid transactions and invalid transactions. A valid amount can be converted to a number. Save the report as JSON.

**Hint:** Each list can contain dictionaries. Be careful to store JSON-compatible values.


In [ ]:
# Your solution:



## 7. Mini-project — Daily Operations Report

### Scenario
You support an operations team. A CSV file contains a list of network events. Create a report that a team manager can read.

### Requirements
1. Read `network_events.csv`.
2. Calculate the total number of events.
3. Count events by priority.
4. Count events whose status is `Open`.
5. Calculate the average `hours_open` among open events.
6. Save the summary to `daily_operations_report.json`.
7. Print a readable summary in the notebook.

### Acceptance checklist
- [ ] Total events equals the number of input records.
- [ ] Priority counts add up to the total.
- [ ] Open events do not exceed total events.
- [ ] The average is calculated from open events only.
- [ ] The JSON file can be loaded back with `json.load()`.
- [ ] The original input CSV has not been changed.

Start with the skeleton below. Build one requirement at a time.


In [ ]:
events_path = DATA_DIR / "network_events.csv"
project_output_path = OUTPUT_DIR / "daily_operations_report.json"

# Step 1: Read all rows.
# Step 2: Calculate the metrics.
# Step 3: Build a summary dictionary.
# Step 4: Save it with json.dump().
# Step 5: Load the saved JSON again and verify it.



## 8. Exit ticket: can you explain it without copying?

Answer these in your own words before finishing.

1. What is the difference between a file path and a file's contents?
2. Why is `with open(...)` (or `Path.open(...)`) useful?
3. Why are numbers read from CSV often strings?
4. What is the difference between `json.load()` and `json.loads()`?
5. What is the difference between writing and appending to a text file?
6. Which error would you expect when converting `"unknown"` to `float`?
7. Name one way to check whether your output is correct.

### Self-assessment
Rate each statement from 1 (not yet) to 5 (I can do this independently):
- I can read and write a text file.
- I can process CSV rows and calculate a total.
- I can load nested JSON data.
- I can handle a common file or data error.
- I can combine these skills in a small project.

**Next step:** Revisit any item rated 1 or 2, rerun its example, and then solve a similar task without copying the code.
